# I-JEPA pre-training on a Kaggle GPU

Trains one run of `jepa/config.py` on a GPU and writes its outputs in the
same layout as the local project, so the analysis scripts can read them
directly.

**Before running**, in the notebook settings (right-hand panel):
- *Accelerator*: **GPU T4 x1** (or P100);
- *Internet*: **on** (needs a phone-verified Kaggle account), to clone the
  repo and download the dataset.

**To train without keeping the tab open**: *Save Version* -> *Save & Run All
(Commit)*. The outputs then appear in the *Output* tab of that version.

Outputs, in `/kaggle/working/`:
- `checkpoints/<run>/jepa.weights.h5` (saved after every epoch);
- `logs/<run>.log` (read by `result_analysis/plot_loss.py`);
- `<run>.zip` with both, to unzip at the project root.

## 1. Parameters

In [ ]:
RUN = "cifar10_d256"   # a preset of jepa/config.py: cifar10_d128, cifar10_d256, stl10_d128, stl10_d256
BRANCH = "main"
EPOCHS = None          # None = config.epochs (30)

REPO_URL = "https://github.com/maxzaragoza2a/i-jepa-from-scratch.git"
REPO_DIR = "/tmp/i-jepa-from-scratch"   # outside /kaggle/working: the code is not an output
WORK = "/kaggle/working"

## 2. Check the GPU and the libraries

Kaggle's own TensorFlow is kept: reinstalling the version pinned in
`requirements.txt` could break the CUDA setup. The code only needs Keras 3.

In [ ]:
!nvidia-smi

import tensorflow as tf
import keras

print("tensorflow", tf.__version__, "| keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU: set Accelerator to a GPU in the notebook settings."
assert int(keras.__version__.split(".")[0]) >= 3, "The code needs Keras 3."

## 3. Get the code

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

## 4. Dependencies

Only STL-10 needs `tensorflow_datasets` (often already installed on Kaggle).
`importlib_resources` is a dependency tfds needs but does not declare.

In [ ]:
try:
    import tensorflow_datasets  # noqa: F401
except ImportError:
    %pip install -q tensorflow-datasets
%pip install -q importlib_resources

## 5. Smoke test

20 steps, written to `/tmp`. Checks that the run works on this setup and
gives a rough time per step before the full run.

In [ ]:
%%time
!cd {REPO_DIR} && python -u -m jepa.train --run {RUN} --max-steps 20 --out /tmp/smoke

## 6. Full training

The log goes to the notebook and to `logs/<run>.log` at the same time.
The weights are saved after every epoch.

In [ ]:
epochs_arg = f"--epochs {EPOCHS}" if EPOCHS else ""
os.makedirs(f"{WORK}/logs", exist_ok=True)

!cd {REPO_DIR} && python -u -m jepa.train --run {RUN} {epochs_arg} --out {WORK}/checkpoints/{RUN} 2>&1 | tee {WORK}/logs/{RUN}.log

## 7. Package the outputs

In [ ]:
!cd {WORK} && rm -f {RUN}.zip && zip -r {RUN}.zip checkpoints/{RUN} logs/{RUN}.log
!ls -lh {WORK}/{RUN}.zip

## 8. Back on the local machine

Download `<run>.zip` from the *Output* tab and unzip it at the project root:
it gives `checkpoints/<run>/jepa.weights.h5` and `logs/<run>.log`. Then:

```bash
python result_analysis/plot_loss.py --run cifar10_d256
python result_analysis/check_collapse.py --run cifar10_d256
python result_analysis/predicted_fill.py --run cifar10_d256
```